# REACT Algorithm Evaluation

Developer notebook for analyzing a REACT optimization run.
Produces:
- **KPI comparison** — before vs after, with delta
- **Gantt 0** — raw API input (inferred timings, only on local runs)
- **Gantt 1** — preassigned schedule before the solver
- **Gantt 2** — new + reassigned labors on an isolated canvas
- **Gantt 3** — full schedule with 5-category colors
- **Detail table** — non-frozen labors with driver before/after

### Freeze reconstruction
Frozen labors are not flagged in the output payload.  
They are reconstructed post-hoc:
- `decision_time` = `run.json.started_at`
- `freeze_cutoff` = `decision_time + time_previous_freeze minutes`
- A labor is **frozen** if it was preassigned AND `schedule_date ≤ freeze_cutoff`
- A labor is **reassigned** if reassignable AND final driver ≠ original driver
- A labor is **unchanged** if reassignable AND final driver = original driver
- A labor is **new** if it was not in the preassigned schedule

### Color legend

| Color | Category |
|-------|----------|
| 🔵 Navy | Frozen labor |
| 🔴 Red | Reassigned labor |
| 🩵 Light blue | Unchanged (reassignable, kept same driver) |
| 🟢 Green | New labor |
| ⬜ Grey | Free time |

In [ ]:
from __future__ import annotations
import json
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from alfred.analysis.solution_evaluation import (
    load_payload,
    flatten_labors,
    build_preassigned_lookup,
    reconstruct_timeline,
    compute_payload_summary,
    build_gantt_figure,
    build_service_distance_figure,
    build_driver_distance_figure,
    _parse_dt,
    BOGOTA_TZ,
)
from alfred.optimization.settings.model_params import ModelParams
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

_PROJECT_ROOT = Path("../..").resolve()
_params       = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh
GRACE_MINUTES:    int   = _params.tiempo_gracia_min
print(f"speed={ALFRED_SPEED_KMH} km/h | grace={GRACE_MINUTES} min")

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def csv_rows_to_analysis_rows(df: pd.DataFrame) -> list:
    """Convert preassigned_df.csv or output.csv rows to flat dicts for
    reconstruct_timeline / compute_payload_summary."""
    rows = []
    for _, r in df.iterrows():
        actual_start = _parse_dt(str(r.get("actual_start") or ""))
        actual_end   = _parse_dt(str(r.get("actual_end")   or ""))
        duration_min = 0.0
        if actual_start and actual_end:
            duration_min = (actual_end - actual_start).total_seconds() / 60.0
        driver_id = r.get("assigned_driver")
        rows.append({
            "labor_id":                str(r.get("labor_id")),
            "service_id":              r.get("service_id"),
            "labor_type":              r.get("labor_type"),
            "labor_schedule_date":     r.get("schedule_date"),
            "service_labor_index":     r.get("labor_sequence", 0) or 0,
            "driver_id":               str(driver_id) if pd.notna(driver_id) else None,
            "actual_start":            actual_start,
            "actual_end":              actual_end,
            "duration_min":            duration_min,
            "labor_distance_km":       float(r.get("labor_distance_km") or 0.0),
            "driver_move_distance_km": float(r.get("driver_move_distance_km") or 0.0),
            "is_infeasible":           bool(r.get("is_infeasible", False)),
            "reassignment_candidate":  bool(r.get("reassignment_candidate", False)),
            "original_assigned_driver": r.get("assigned_driver"),
            "map_start_wkt":           r.get("map_start_point") or None,
            "map_end_wkt":             r.get("map_end_point")   or None,
        })
    return rows


# ── Multi-color REACT Gantt ───────────────────────────────────────────────────

_REACT_COLORS = {
    "FREE_TIME":       "rgba(190, 190, 190, 0.35)",
    "frozen_move":     "rgba( 30,  60, 130, 0.60)",
    "reassigned_move": "rgba(220,  80,  40, 0.75)",
    "unchanged_move":  "rgba(100, 170, 220, 0.60)",
    "new_move":        "rgba(120, 180,  80, 0.75)",
    "frozen_labor":    "rgba( 20,  40, 120, 0.90)",
    "reassigned_labor":"rgba(200,  50,  30, 0.90)",
    "unchanged_labor": "rgba( 80, 160, 220, 0.88)",
    "new_labor":       "rgba( 40, 170,  80, 0.90)",
}
_REACT_LABELS = {
    "FREE_TIME":       "Free time",
    "frozen_move":     "Driver move (frozen)",
    "reassigned_move": "Driver move (reassigned)",
    "unchanged_move":  "Driver move (unchanged)",
    "new_move":        "Driver move (new)",
    "frozen_labor":    "Labor — frozen",
    "reassigned_labor":"Labor — reassigned",
    "unchanged_labor": "Labor — unchanged (kept)",
    "new_labor":       "Labor — new",
}
_REACT_ORDER = [
    "FREE_TIME",
    "frozen_move", "reassigned_move", "unchanged_move", "new_move",
    "frozen_labor", "reassigned_labor", "unchanged_labor", "new_labor",
]
_MIN_VT_MS = 5 * 60 * 1000


def _to_plotly_dt(ts) -> str:
    if ts is None:
        return ""
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        ts = ts.tz_convert("America/Bogota").tz_localize(None)
    return ts.isoformat()


def _classify_react(seg_type: str, origin: str) -> str:
    if seg_type == "FREE_TIME":
        return "FREE_TIME"
    if seg_type == "DRIVER_MOVE":
        return f"{origin}_move"
    return f"{origin}_labor"


def build_react_gantt(segments_with_origin: list, driver_ids: list,
                      label: str = "REACT") -> go.Figure:
    """5-color Gantt for REACT. Segments must have an 'origin' field."""
    by_type = defaultdict(list)
    drv_set = set(driver_ids)
    for seg in segments_with_origin:
        if seg["driver_id"] in drv_set:
            key = _classify_react(seg["segment_type"], seg.get("origin", "unchanged"))
            if key in _REACT_COLORS:
                by_type[key].append(seg)

    fig = go.Figure()
    for seg_type in _REACT_ORDER:
        segs = by_type.get(seg_type, [])
        if not segs:
            continue
        base_vals, x_vals, y_vals, hovers = [], [], [], []
        for s in segs:
            dur_ms = (s["end"] - s["start"]).total_seconds() * 1000
            if dur_ms < 0:
                continue
            is_labor = seg_type.endswith("_labor")
            if is_labor and dur_ms == 0:
                dur_ms = _MIN_VT_MS
            elif not is_labor and seg_type != "FREE_TIME" and dur_ms == 0:
                continue
            base_vals.append(_to_plotly_dt(s["start"]))
            x_vals.append(dur_ms)
            y_vals.append(s["driver_id"])
            ht = (
                f"<b>{_REACT_LABELS.get(seg_type, seg_type)}</b><br>"
                f"Driver: {s['driver_id']}<br>"
                f"Labor: {s['labor_id']}<br>"
                f"Service: {s['service_id']}<br>"
                f"Start: {pd.Timestamp(s['start']).strftime('%H:%M')}<br>"
                f"End: {pd.Timestamp(s['end']).strftime('%H:%M')}<br>"
                f"Duration: {s['duration_min']:.1f} min"
            )
            if s.get("distance_km", 0) > 0:
                ht += f"<br>Distance: {s['distance_km']:.2f} km"
            if s.get("is_infeasible"):
                ht += "<br><i>⚠ infeasible</i>"
            hovers.append(ht)
        if not x_vals:
            continue
        fig.add_trace(go.Bar(
            x=x_vals, y=y_vals, base=base_vals, orientation="h",
            name=_REACT_LABELS.get(seg_type, seg_type),
            marker_color=_REACT_COLORS[seg_type],
            hovertext=hovers, hoverinfo="text",
            showlegend=True, legendgroup=seg_type,
        ))

    all_segs_in_plot = [s for segs in by_type.values() for s in segs if s["driver_id"] in drv_set]
    pad = timedelta(minutes=30)
    if all_segs_in_plot:
        t_min = min(s["start"] for s in all_segs_in_plot)
        t_max = max(s["end"]   for s in all_segs_in_plot)
        fig.update_xaxes(type="date", range=[_to_plotly_dt(t_min - pad), _to_plotly_dt(t_max + pad)])
    fig.update_xaxes(tickformat="%H:%M", title_text="Time (Bogotá)")
    row_h = max(35, 600 // max(len(driver_ids), 1))
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=list(reversed(driver_ids)),
        title_text="Driver",
    )
    fig.update_layout(
        title_text=label, barmode="overlay",
        height=max(420, row_h * len(driver_ids) + 160),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="center", x=0.5),
        hovermode="closest",
        margin=dict(l=120, r=20, t=100, b=60),
    )
    return fig

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set RUN_DIR to a specific REACT run directory, or leave None to auto-pick.
RUN_DIR: Optional[Path] = None
ALGORITHM_FILTER: str   = "REACT"

# Override freeze minutes if needed (e.g. for older runs that pre-date the fix
# that writes time_previous_freeze into run.json). None = read from run.json.
FREEZE_MINUTES: Optional[int] = None

_runs_base = _PROJECT_ROOT / "data" / "runs"
if RUN_DIR is None:
    _candidates = sorted(
        [
            d for d in _runs_base.iterdir()
            if d.is_dir()
            and (d / "run.json").exists()
            and (d / "output" / "output.csv").exists()
        ],
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    _matching = []
    for _d in _candidates:
        try:
            _mf = json.loads((_d / "run.json").read_text())
            if str(_mf.get("solver", "")).upper() == ALGORITHM_FILTER:
                _matching.append(_d)
        except Exception:
            pass
    if not _matching:
        print(f"[warning] No completed runs found with solver={ALGORITHM_FILTER!r}.")
        print("Available runs:")
        for _d in _candidates[:5]:
            _solver = ""
            try:
                _solver = json.loads((_d / "run.json").read_text()).get("solver", "?")
            except Exception:
                pass
            print(f"  {_d.name}  solver={_solver}")
        print("\nSet RUN_DIR manually to proceed.")
        RUN_DIR = _candidates[0] if _candidates else None
    else:
        RUN_DIR = _matching[0]

if RUN_DIR is None:
    raise FileNotFoundError("No suitable run found. Set RUN_DIR manually.")

LABEL_BEFORE = "before"
LABEL_AFTER  = "after (react)"
print(f"Run dir : {RUN_DIR}")

In [ ]:
# ── Load manifest and extract freeze parameters ───────────────────────────────
_manifest = json.loads((RUN_DIR / "run.json").read_text())
print(json.dumps(_manifest, indent=2, default=str))

_algo_cfg = _manifest.get("algorithm") or {}
if isinstance(_algo_cfg, str):
    _algo_cfg = {"name": _algo_cfg}

# Resolve freeze minutes: manual override > run.json > default 0
if FREEZE_MINUTES is not None:
    _freeze_minutes = FREEZE_MINUTES
    print(f"[config] Using manual FREEZE_MINUTES={_freeze_minutes}")
else:
    _tpf = _algo_cfg.get("time_previous_freeze")
    if _tpf is not None:
        _freeze_minutes = int(_tpf)
        print(f"[config] FREEZE_MINUTES={_freeze_minutes} (from run.json)")
    else:
        print("[warning] time_previous_freeze not found in run.json.")
        print("          Set FREEZE_MINUTES manually if this is a REACT run.")
        print("          Defaulting to 0 — no labors will be classified as frozen.")
        _freeze_minutes = 0

# Decision time: run started_at as proxy
_started_at_raw = _manifest.get("started_at")
if _started_at_raw:
    _decision_time = pd.Timestamp(_started_at_raw)
    if _decision_time.tzinfo is None:
        _decision_time = _decision_time.tz_localize("America/Bogota")
    else:
        _decision_time = _decision_time.tz_convert("America/Bogota")
else:
    print("[warning] started_at not in run.json — using current time as decision_time.")
    _decision_time = pd.Timestamp.now(tz="America/Bogota")

_freeze_cutoff = _decision_time + pd.Timedelta(minutes=_freeze_minutes)
print(f"\ndecision_time : {_decision_time}")
print(f"freeze_minutes: {_freeze_minutes}")
print(f"freeze_cutoff : {_freeze_cutoff}")

In [ ]:
# ── Load CSV files ────────────────────────────────────────────────────────────
_pre_csv = RUN_DIR / "intermediate" / "preassigned_df.csv"
_out_csv = RUN_DIR / "output" / "output.csv"

_pre_df = pd.read_csv(_pre_csv, low_memory=False) if _pre_csv.exists() else pd.DataFrame()
_out_df = pd.read_csv(_out_csv, low_memory=False)

for _df in (_pre_df, _out_df):
    for _col in ("actual_start", "actual_end", "schedule_date"):
        if _col in _df.columns:
            _df[_col] = _df[_col].apply(lambda x: pd.Timestamp(x) if pd.notna(x) else pd.NaT)

if _pre_df.empty or "labor_id" not in _pre_df.columns:
    print("[warning] preassigned_df is empty — 'Before' sections will be skipped.")

print(f"preassigned_df : {len(_pre_df)} rows")
print(f"output.csv     : {len(_out_df)} rows")

In [ ]:
# ── Freeze classification ─────────────────────────────────────────────────────
# Build lookups from preassigned_df
_pre_assignment: dict = {}   # labor_id -> original driver
_pre_schedule:   dict = {}   # labor_id -> schedule_date

if not _pre_df.empty and "labor_id" in _pre_df.columns:
    for _, _row in _pre_df.iterrows():
        _lid = str(_row["labor_id"])
        _pre_assignment[_lid] = _row.get("assigned_driver")
        _pre_schedule[_lid]   = _row.get("schedule_date")

_final_assignment: dict = {}
for _, _row in _out_df.iterrows():
    _final_assignment[str(_row["labor_id"])] = _row.get("assigned_driver")

_preassigned_labor_ids = set(_pre_assignment.keys())


def _driver_str(v) -> str:
    return str(v) if pd.notna(v) else "__none__"


def _classify_react_origin(labor_id: str) -> str:
    """Classify a labor as frozen / reassigned / unchanged / new."""
    lid = str(labor_id)
    if lid not in _preassigned_labor_ids:
        return "new"

    sched = _pre_schedule.get(lid)
    if sched is None or pd.isna(sched):
        return "unchanged"   # conservative — no schedule date to compare

    if isinstance(sched, pd.Timestamp):
        if sched.tzinfo is None:
            sched = sched.tz_localize("America/Bogota")
        else:
            sched = sched.tz_convert("America/Bogota")

    if sched <= _freeze_cutoff:
        return "frozen"

    orig  = _driver_str(_pre_assignment.get(lid))
    final = _driver_str(_final_assignment.get(lid))
    return "reassigned" if orig != final else "unchanged"


_origin_map: dict = {}
for _lid in _preassigned_labor_ids:
    _origin_map[_lid] = _classify_react_origin(_lid)
for _lid in _final_assignment:
    if _lid not in _origin_map:
        _origin_map[_lid] = "new"

from collections import Counter
_counts = Counter(_origin_map.values())
print("Labor classification:")
for _cat in ("frozen", "reassigned", "unchanged", "new"):
    print(f"  {_cat:12s}: {_counts.get(_cat, 0)}")

In [ ]:
# ── Build row dicts and reconstruct timelines ─────────────────────────────────
rows_before = csv_rows_to_analysis_rows(_pre_df) if not _pre_df.empty else []
rows_after  = csv_rows_to_analysis_rows(_out_df)

for r in rows_before:
    r["origin"] = "preassigned"
for r in rows_after:
    r["origin"] = _origin_map.get(r["labor_id"], "new")

segments_before = reconstruct_timeline(rows_before, ALFRED_SPEED_KMH) if rows_before else []
segments_after  = reconstruct_timeline(rows_after,  ALFRED_SPEED_KMH)

for seg in segments_before:
    seg["origin"] = "preassigned"
for seg in segments_after:
    seg["origin"] = _origin_map.get(seg["labor_id"], "new")

drivers_before = sorted({r["driver_id"] for r in rows_before if r["driver_id"]})
drivers_after  = sorted({r["driver_id"] for r in rows_after  if r["driver_id"]})
all_services   = sorted({str(r["service_id"]) for r in rows_before + rows_after if r["service_id"] is not None})

print(f"segments_before : {len(segments_before)} | drivers: {len(drivers_before)}")
print(f"segments_after  : {len(segments_after)} | drivers: {len(drivers_after)}")

---
## Gantt 0 — Raw API Input (Inferred Timings)
_Only available on local runs where `input/processed_input.json` was saved._  
_Timings are inferred from `estimated_time` — not solver-computed._

In [ ]:
# ── Raw input Gantt (optional) ────────────────────────────────────────────────
_raw_input_path = RUN_DIR / "input" / "processed_input.json"

if not _raw_input_path.exists():
    print("[info] processed_input.json not found (API run) — Gantt 0 skipped.")
else:
    _raw_services   = load_payload(_raw_input_path)
    _raw_pre_lookup = build_preassigned_lookup(_raw_services)

    _raw_rows, _ = flatten_labors(_raw_services, preassigned_lookup=_raw_pre_lookup)

    _estimated_times = {
        labor.get("id"): labor.get("estimated_time") or 0
        for svc in _raw_services
        for labor in (svc.get("service_labors") or svc.get("serviceLabors") or [])
    }
    for r in _raw_rows:
        if r["actual_start"] and r["duration_min"] == 0:
            est = _estimated_times.get(r["labor_id"]) or 0
            r["duration_min"] = float(est)
            if est > 0:
                r["actual_end"] = r["actual_start"] + timedelta(minutes=float(est))

    _raw_drivers = sorted({r["driver_id"] for r in _raw_rows if r["driver_id"]})
    if _raw_drivers:
        _raw_segs = reconstruct_timeline(_raw_rows, ALFRED_SPEED_KMH)
        build_gantt_figure(_raw_segs, _raw_drivers,
                           f"Raw Input (schedule_date, estimated durations) — {RUN_DIR.name}").show()
    else:
        print("[info] Raw input has no preassigned labors with driver — nothing to display.")

---
## KPI Comparison — Before vs After

In [ ]:
# ── KPI comparison ────────────────────────────────────────────────────────────
_kpi_before = compute_payload_summary(
    rows_before, tiempo_gracia_min=GRACE_MINUTES, segments=segments_before
) if rows_before else {}
_kpi_after = compute_payload_summary(
    rows_after, tiempo_gracia_min=GRACE_MINUTES, segments=segments_after
)

def _delta(a, b):
    try:
        return round(float(b) - float(a), 4)
    except (TypeError, ValueError):
        return None

def _delta_pct(a, b):
    try:
        fa, fb = float(a), float(b)
        return round(100.0 * (fb - fa) / fa, 2) if fa != 0 else None
    except (TypeError, ValueError):
        return None

_METRICS = [
    ("services_count",                "services"),
    ("labors_count",                  "labors_total"),
    ("labors_vt_count",               "labors_vt"),
    ("drivers_count",                 "drivers_used"),
    ("labors_assigned",               "labors_assigned"),
    ("labors_preassigned",            "labors_preassigned"),
    ("labors_infeasible",             "labors_infeasible"),
    ("labors_infeasible_pct",         "labors_infeasible_pct"),
    ("labors_in_grace",               "labors_in_grace"),
    ("total_grace_min",               "total_grace_min"),
    ("total_labor_distance_km",       "total_labor_distance_km"),
    ("total_driver_move_distance_km", "total_driver_move_distance_km"),
    ("total_distance_km",             "total_distance_km"),
    ("avg_labor_distance_km",         "avg_labor_distance_km"),
    ("avg_driver_move_distance_km",   "avg_driver_move_distance_km"),
]

pd.DataFrame([
    {
        "metric":      label,
        LABEL_BEFORE:  _kpi_before.get(key),
        LABEL_AFTER:   _kpi_after.get(key),
        "delta":       _delta(_kpi_before.get(key), _kpi_after.get(key)),
        "delta_pct":   _delta_pct(_kpi_before.get(key), _kpi_after.get(key)),
    }
    for key, label in _METRICS
]).set_index("metric")

In [ ]:
# ── REACT-specific KPI summary ────────────────────────────────────────────────
pd.DataFrame([
    {"metric": "frozen_labors",     "value": _counts.get("frozen", 0)},
    {"metric": "reassigned_labors", "value": _counts.get("reassigned", 0)},
    {"metric": "unchanged_labors",  "value": _counts.get("unchanged", 0)},
    {"metric": "new_labors",        "value": _counts.get("new", 0)},
    {"metric": "freeze_minutes",    "value": _freeze_minutes},
    {"metric": "decision_time",     "value": str(_decision_time)},
    {"metric": "freeze_cutoff",     "value": str(_freeze_cutoff)},
]).set_index("metric")

---
## Gantt 1 — Before (Preassigned Schedule)

In [ ]:
if rows_before:
    build_gantt_figure(
        segments_before, drivers_before,
        f"Before — preassigned schedule — {RUN_DIR.name}",
    ).show()
else:
    print("[info] No preassigned schedule to display (preassigned_df is empty).")

---
## Gantt 2 — New and Reassigned Labors (Isolated)
_Each labor shown on its own virtual driver row — no move context._

In [ ]:
_changed_rows = [
    r for r in rows_after if r["origin"] in ("new", "reassigned")
]

if _changed_rows:
    _virtual_rows = []
    for r in sorted(_changed_rows, key=lambda x: (x["actual_start"] or datetime.min)):
        _vrow = dict(r)
        _vrow["driver_id"]               = f"{r['origin'].upper()}:{r['labor_id']}"
        _vrow["driver_move_distance_km"]  = 0.0
        _virtual_rows.append(_vrow)

    _virtual_segs = reconstruct_timeline(_virtual_rows, ALFRED_SPEED_KMH)
    _vt_segs      = [s for s in _virtual_segs if s["segment_type"] == "VEHICLE_TRANSPORTATION"]
    _virtual_drv  = [r["driver_id"] for r in _virtual_rows]

    build_gantt_figure(
        _vt_segs, _virtual_drv,
        f"New + Reassigned (isolated) — {RUN_DIR.name}",
    ).show()
else:
    print("[info] No new or reassigned labors found.")

---
## Gantt 3 — Full Schedule (5-Color by REACT Category)

| Color | Category |
|-------|----------|
| 🔵 Navy | Frozen labor |
| 🔴 Red | Reassigned labor |
| 🩵 Light blue | Unchanged (reassignable, kept same driver) |
| 🟢 Green | New labor |
| ⬜ Grey | Free time |

In [ ]:
build_react_gantt(
    segments_after, drivers_after,
    label=f"Full Schedule (REACT) — {RUN_DIR.name}",
).show()

---
## Distance per Service / Driver

In [ ]:
build_service_distance_figure(rows_after, all_services, LABEL_AFTER).show()
build_driver_distance_figure(rows_after, drivers_after, LABEL_AFTER).show()

---
## Non-Frozen Labor Detail
_Frozen labors are omitted (no change happened to them)._

In [ ]:
_non_frozen_ids = {
    lid for lid, origin in _origin_map.items() if origin != "frozen"
}

_detail_df = _out_df[_out_df["labor_id"].astype(str).isin(_non_frozen_ids)].copy()
_detail_df["origin"]         = _detail_df["labor_id"].astype(str).map(_origin_map).fillna("new")
_detail_df["driver_before"]  = _detail_df["labor_id"].astype(str).map(
    lambda lid: _pre_assignment.get(lid)
)
_detail_df["driver_after"]   = _detail_df["assigned_driver"]
_detail_df["driver_changed"] = (
    _detail_df["driver_before"].apply(_driver_str)
    != _detail_df["driver_after"].apply(_driver_str)
)

_detail_cols = [c for c in [
    "service_id", "labor_id", "origin",
    "driver_before", "driver_after", "driver_changed",
    "schedule_date", "actual_start", "actual_end",
    "is_infeasible", "infeasibility_cause_code",
    "labor_distance_km",
] if c in _detail_df.columns]

if _detail_df.empty:
    print("[info] No non-frozen labors to display.")
else:
    _detail_df[_detail_cols].sort_values(
        ["origin", "driver_after", "actual_start"]
    ).reset_index(drop=True)